In [3]:
# CELL 1: Install and import required libraries, and create project directories
import sys
import os

# Install required packages quietly
!{sys.executable} -m pip install -q easyocr spacy numpy torch pandas

import easyocr
import spacy
import torch
import numpy as np
import pandas as pd

# Create folders to organize our saved model state, input images, and untranslated text dumps
os.makedirs("model_storage", exist_ok=True)
os.makedirs("input_images", exist_ok=True)
os.makedirs("untranslated_dumps", exist_ok=True)

# Download English NLP model for our Part-of-Speech (POS) classification layer later
try:
    nlp = spacy.load("en_core_web_sm")
    print("✓ Spacy English model loaded successfully.")
except OSError:
    print("Downloading Spacy English model...")
    !{sys.executable} -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm")

print("✓ All libraries imported.")
print("✓ Directories created: /model_storage, /input_images, /untranslated_dumps")

✓ Spacy English model loaded successfully.
✓ All libraries imported.
✓ Directories created: /model_storage, /input_images, /untranslated_dumps


In [4]:
# CELL 2: Core Memory, Unlearning Protocol, and POS Classification Engine
import json

class IncrementalLanguageEngine:
    def __init__(self, storage_path="model_storage/language_engine.pt"):
        self.storage_path = storage_path
        # Core memory structures
        self.dictionary = {}      # Format: {unknown_word: english_translation}
        self.weights = {}         # Format: {unknown_word: weight_float}
        self.pos_tags = {}        # Format: {unknown_word: pos_category_string}

        # Load existing state if it exists
        self.load_model()

    def learn_word(self, unknown_word, english_translation, pos_category=None):
        """Learns or updates a word translation pair with full weight."""
        unknown_word = unknown_word.strip().lower()
        english_translation = english_translation.strip().lower()

        self.dictionary[unknown_word] = english_translation
        self.weights[unknown_word] = 1.0  # Set active weight

        # Auto-classify POS if not explicitly provided
        if not pos_category:
            doc = nlp(english_translation)
            pos_category = doc[0].pos_ if len(doc) > 0 else "UNKNOWN"

        self.pos_tags[unknown_word] = pos_category
        self.save_model()

    def unlearn_word(self, unknown_word):
        """Requirement #3: Instantly zeroes out the weight to make the model unlearn a mistake."""
        unknown_word = unknown_word.strip().lower()
        if unknown_word in self.weights:
            self.weights[unknown_word] = 0.0
            # Retain the key structural slot, but clear the translation mapping
            self.dictionary[unknown_word] = "[UNLEARNED_WAITING_CORRECTION]"
            self.pos_tags[unknown_word] = "UNKNOWN"
            self.save_model()
            return True
        return False

    def translate_token(self, unknown_word):
        """Translates a token ONLY if it has been learned and its weight is greater than zero."""
        unknown_word = unknown_word.strip().lower()
        if unknown_word in self.dictionary and self.weights.get(unknown_word, 0.0) > 0.0:
            return self.dictionary[unknown_word]
        return None

    def save_model(self):
        """Requirement #1: Saves the entire memory architecture as a reusable model file."""
        state = {
            "dictionary": self.dictionary,
            "weights": self.weights,
            "pos_tags": self.pos_tags
        }
        torch.save(state, self.storage_path)

    def load_model(self):
        """Loads the saved state from disk if it exists."""
        if os.path.exists(self.storage_path):
            try:
                state = torch.load(self.storage_path, weights_only=False)
                self.dictionary = state.get("dictionary", {})
                self.weights = state.get("weights", {})
                self.pos_tags = state.get("pos_tags", {})
                print(f"✓ Model successfully loaded from {self.storage_path}")
                print(f"  Current Vocabulary Size: {len(self.dictionary)} words.")
            except Exception as e:
                print(f"Creating new model state (Could not read existing file: {e})")
        else:
            print("Initialized fresh, empty model state.")

# --- Test the Logic In-Place ---
print("Testing Engine Initialization...")
engine = IncrementalLanguageEngine()

# Test Learning & POS Categorization (Requirement #4)
engine.learn_word("koshur", "kashmiri", pos_category="PROPN")
engine.learn_word("khyon", "eat") # Should auto-detect as VERB

print(f"\nTest 1 (Translation check): 'khyon' -> {engine.translate_token('khyon')} ({engine.pos_tags.get('khyon')})")

# Test Instant Unlearning (Requirement #3)
print("\nSimulating a mistake correction...")
engine.learn_word("bad_token", "wrong_meaning")
print(f"Before unlearning: 'bad_token' -> {engine.translate_token('bad_token')}, Weight: {engine.weights.get('bad_token')}")

engine.unlearn_word("bad_token")
print(f"After zeroing weight: 'bad_token' -> {engine.translate_token('bad_token')}, Weight: {engine.weights.get('bad_token')}")

Testing Engine Initialization...
✓ Model successfully loaded from model_storage/language_engine.pt
  Current Vocabulary Size: 3 words.

Test 1 (Translation check): 'khyon' -> eat (VERB)

Simulating a mistake correction...
Before unlearning: 'bad_token' -> wrong_meaning, Weight: 1.0
After zeroing weight: 'bad_token' -> None, Weight: 0.0


In [5]:
# CELL 3: Optical Character Recognition & Untranslated Text Batcher
class ImageLanguageProcessor:
    def __init__(self, language_engine):
        self.engine = language_engine
        # Initialize EasyOCR Reader for general character recognition
        print("Loading OCR Models...")
        self.reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
        print("✓ OCR System initialized.")

    def process_page(self, image_path, page_id="page_01"):
        """Requirement #2: Translates familiar text, compiles layout, bundles unknowns to text files."""
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found at path: {image_path}")

        print(f"\nProcessing Image: {image_path}...")
        ocr_results = self.reader.readtext(image_path)

        translated_lines = []
        unknown_tokens = set()

        for bounding_box, text, confidence in ocr_results:
            clean_token = text.strip().lower()
            if not clean_token:
                continue

            # Attempt engine translation check
            known_translation = self.engine.translate_token(clean_token)

            if known_translation:
                translated_lines.append(f"{known_translation} (orig: {text})")
            else:
                translated_lines.append(f"[{text}]")
                unknown_tokens.add(clean_token)

        # Compile full visual layout summary
        compiled_page_text = " ".join(translated_lines)

        # Save structural batch text file for missing data if unknowns exist
        dump_path = f"untranslated_dumps/needed_translations_{page_id}.txt"
        if unknown_tokens:
            with open(dump_path, "w", encoding="utf-8") as f:
                f.write(f"# Translation required for page: {page_id}\n")
                f.write("# Format: unknown_word = english_translation\n\n")
                for token in sorted(unknown_tokens):
                    f.write(f"{token} = \n")
            print(f"⚠️ {len(unknown_tokens)} unknown tokens found. Saved template to: {dump_path}")
        else:
            print("✓ Perfect translation match! Zero unknown tokens remaining on this page.")

        return compiled_page_text, dump_path

# Instantiate the pipeline component
processor = ImageLanguageProcessor(engine)

Loading OCR Models...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✓ OCR System initialized.


In [6]:
# CELL 4: Interactive Update Loop, Corrections Manager, and Data Inspectors
import re

def feed_batch_translations(dump_file_path):
    """Parses your edited txt file, extracts translations, and feeds them into the engine memory."""
    if not os.path.exists(dump_file_path):
        print(f"❌ File not found: {dump_file_path}")
        return

    print(f"Reading translations from {dump_file_path}...")
    learned_count = 0

    with open(dump_file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            # Skip empty lines and comment headers
            if not line or line.startswith("#"):
                continue

            if "=" in line:
                parts = line.split("=", 1)
                unknown_word = parts[0].strip().lower()
                english_translation = parts[1].strip().lower()

                # Only add if you provided a translation after the '=' sign
                if english_translation:
                    engine.learn_word(unknown_word, english_translation)
                    print(f"  → Learned: '{unknown_word}' = '{english_translation}' [{engine.pos_tags[unknown_word]}]")
                    learned_count += 1

    print(f"\n✓ Successfully ingested {learned_count} new words into the library storage!")

def correct_mistake(wrong_unknown_word, correct_english_translation):
    """Requirement #3: Forcefully unlearns an entry by zero-weighting it, then maps the correct one."""
    wrong_unknown_word = wrong_unknown_word.strip().lower()

    print(f"\n[CORRECTION PROCESS] Flagging error for token: '{wrong_unknown_word}'")
    # Step 1: Zero out weight completely to erase memory trace
    engine.unlearn_word(wrong_unknown_word)
    print(f"  → Memory weight zeroed out. Old mapping deleted.")

    # Step 2: Inject the new valid translation state
    engine.learn_word(wrong_unknown_word, correct_english_translation)
    print(f"  → Reinjected correct pair: '{wrong_unknown_word}' = '{correct_english_translation}' [{engine.pos_tags[wrong_unknown_word]}]")
    print(f"✓ Engine database updated and saved.")

def inspect_library():
    """Prints a neat breakdown of our currently learned language vocabulary."""
    if not engine.dictionary:
        print("Your language model library is currently empty.")
        return

    data = []
    for word in engine.dictionary:
        data.append({
            "Unknown Word": word,
            "English Translation": engine.dictionary[word],
            "Memory Weight": engine.weights.get(word, 0.0),
            "Part of Speech": engine.pos_tags.get(word, "UNKNOWN")
        })
    df = pd.DataFrame(data)
    print("\n--- CURRENT ENGINE LIBRARY STATUS ---")
    print(df.to_string(index=False))

print("✓ Runtime framework loaded.")

✓ Runtime framework loaded.


In [7]:
# CELL 5: Complete End-to-End Test and Simulation Run
import cv2

def create_mock_target_image(filename="input_images/unknown_page_01.png"):
    """Generates a clean mock image containing dummy unknown language words for testing."""
    # Create a simple white background image using numpy arrays
    img = np.ones((150, 600, 3), dtype=np.uint8) * 255

    # We write a sentence in our simulated unknown language:
    # "Koshur lukh khyon tsot" (Meaning: Kashmiri people eat bread)
    text_to_render = "Koshur lukh khyon tsot"

    # Draw text onto our canvas using OpenCV
    cv2.putText(img, text_to_render, (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)
    cv2.imwrite(filename, img)
    print(f"✓ Synthetic test image created at: {filename}")

# --- EXECUTE TRIAL RUN PIPELINE ---

# 1. Create a dummy test image
create_mock_target_image()

# 2. First Pass: The engine scans the image with zero vocabulary context
print("\n=== RUNNING INITIAL IMAGE PASS (NO DICTIONARY CONTEXT) ===")
initial_output, initial_dump = processor.process_page("input_images/unknown_page_01.png", page_id="page_01")
print(f"\nModel Output Layout:\n{initial_output}")

# 3. Simulating User Feedback Loop
# Instead of opening the text file manually, we simulate writing translations to it programmatically:
print("\n=== SIMULATING USER EDITING THE TEMPLATE FILE ===")
with open(initial_dump, "w", encoding="utf-8") as f:
    f.write("koshur = kashmiri\n")
    f.write("lukh = people\n")
    f.write("khyon = eat\n")
    f.write("tsot = bread\n")

# Feed the updated translations back to our saved library
feed_batch_translations(initial_dump)

# 4. Second Pass: The engine processes the same image again now that it has learned
print("\n=== RUNNING SECOND PASS (WITH NEW VOCABULARY MEMORY) ===")
smart_output, _ = processor.process_page("input_images/unknown_page_01.png", page_id="page_01")
print(f"\nSmart Model Output Layout:\n{smart_output}")

# 5. Check out our classified language library
inspect_library()

print("\n=======================================================")
print("✓ End-to-End framework test complete!")

✓ Synthetic test image created at: input_images/unknown_page_01.png

=== RUNNING INITIAL IMAGE PASS (NO DICTIONARY CONTEXT) ===

Processing Image: input_images/unknown_page_01.png...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


⚠️ 1 unknown tokens found. Saved template to: untranslated_dumps/needed_translations_page_01.txt

Model Output Layout:
[Koshur Iukh khyon tsot]

=== SIMULATING USER EDITING THE TEMPLATE FILE ===
Reading translations from untranslated_dumps/needed_translations_page_01.txt...
  → Learned: 'koshur' = 'kashmiri' [PROPN]
  → Learned: 'lukh' = 'people' [NOUN]
  → Learned: 'khyon' = 'eat' [VERB]
  → Learned: 'tsot' = 'bread' [NOUN]

✓ Successfully ingested 4 new words into the library storage!

=== RUNNING SECOND PASS (WITH NEW VOCABULARY MEMORY) ===

Processing Image: input_images/unknown_page_01.png...
⚠️ 1 unknown tokens found. Saved template to: untranslated_dumps/needed_translations_page_01.txt

Smart Model Output Layout:
[Koshur Iukh khyon tsot]

--- CURRENT ENGINE LIBRARY STATUS ---
Unknown Word            English Translation  Memory Weight Part of Speech
      koshur                       kashmiri            1.0          PROPN
       khyon                            eat            1.0

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# # CELL 6: Compiling and Exporting Your Reusable Library Module
# code_content = """# Reusable Incremental Language Engine Library
# import os
# import easyocr
# import spacy
# import torch
# import numpy as np
# import pandas as pd

# class IncrementalLanguageEngine:
#     def __init__(self, storage_path="model_storage/language_engine.pt"):
#         self.storage_path = storage_path
#         self.dictionary = {}
#         self.weights = {}
#         self.pos_tags = {}

#         # Load English spacy for default POS tagging
#         try:
#             self.nlp = spacy.load("en_core_web_sm")
#         except OSError:
#             import sys
#             os.system(f"{sys.executable} -m spacy download en_core_web_sm")
#             self.nlp = spacy.load("en_core_web_sm")

#         self.load_model()

#     def learn_word(self, unknown_word, english_translation, pos_category=None):
#         unknown_word = unknown_word.strip().lower()
#         english_translation = english_translation.strip().lower()

#         self.dictionary[unknown_word] = english_translation
#         self.weights[unknown_word] = 1.0

#         if not pos_category:
#             doc = self.nlp(english_translation)
#             pos_category = doc[0].pos_ if len(doc) > 0 else "UNKNOWN"

#         self.pos_tags[unknown_word] = pos_category
#         self.save_model()

#     def unlearn_word(self, unknown_word):
#         unknown_word = unknown_word.strip().lower()
#         if unknown_word in self.weights:
#             self.weights[unknown_word] = 0.0
#             self.dictionary[unknown_word] = "[UNLEARNED_WAITING_CORRECTION]"
#             self.pos_tags[unknown_word] = "UNKNOWN"
#             self.save_model()
#             return True
#         return False

#     def translate_token(self, unknown_word):
#         unknown_word = unknown_word.strip().lower()
#         if unknown_word in self.dictionary and self.weights.get(unknown_word, 0.0) > 0.0:
#             return self.dictionary[unknown_word]
#         return None

#     def save_model(self):
#         state = {"dictionary": self.dictionary, "weights": self.weights, "pos_tags": self.pos_tags}
#         torch.save(state, self.storage_path)

#     def load_model(self):
#         if os.path.exists(self.storage_path):
#             state = torch.load(self.storage_path, weights_only=False)
#             self.dictionary = state.get("dictionary", {})
#             self.weights = state.get("weights", {})
#             self.pos_tags = state.get("pos_tags", {})

# class ImageLanguageProcessor:
#     def __init__(self, language_engine):
#         self.engine = language_engine
#         self.reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

#     def process_page(self, image_path, page_id="page_01"):
#         if not os.path.exists(image_path):
#             raise FileNotFoundError(f"Image not found at path: {image_path}")

#         ocr_results = self.reader.readtext(image_path)
#         translated_lines = []
#         unknown_tokens = set()

#         for bounding_box, text, confidence in ocr_results:
#             clean_token = text.strip().lower()
#             if not clean_token:
#                 continue

#             known_translation = self.engine.translate_token(clean_token)
#             if known_translation:
#                 translated_lines.append(f"{known_translation}")
#             else:
#                 translated_lines.append(f"[{text}]")
#                 unknown_tokens.add(clean_token)

#         compiled_page_text = " ".join(translated_lines)
#         dump_path = f"untranslated_dumps/needed_translations_{page_id}.txt"

#         if unknown_tokens:
#             with open(dump_path, "w", encoding="utf-8") as f:
#                 f.write(f"# Translation required for page: {page_id}\\n\\n")
#                 for token in sorted(unknown_tokens):
#                     f.write(f"{token} = \\n")

#         return compiled_page_text, dump_path
# """

# with open("incremental_engine.py", "w", encoding="utf-8") as f:
#     f.write(code_content)

# print("✓ Library packaged! 'incremental_engine.py' is saved and ready for production use.")